<a href="https://colab.research.google.com/github/rudraroy1555/resume-screener-ranking/blob/main/tf_idf_resume_screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Candidate Ranking Engine via NLP
**Objective:** An unsupervised model to rank candidate resumes against a specific job description.

In [5]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
from google.colab import drive
drive.mount('/content/drive')

#Target Job Description
job_desc = """
Looking for a software engineer with strong Python skills.
Must have experience with data analysis, pandas, and machine learning.
"""
#Loding data
try:
    df = pd.read_csv("/content/drive/MyDrive/candidates_input.csv")
    print("--- Job Description ---")
    print(job_desc.strip())
    print(f"\n--- Successfully loaded {len(df)} candidates from Drive ---")
    print(df.head())
except :
    print(f"ERROR: Could not find '{"/content/drive/MyDrive/candidates_input.csv"}'.")
    print("Please ensure the file is in your Drive and named correctly.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Job Description ---
Looking for a software engineer with strong Python skills.
Must have experience with data analysis, pandas, and machine learning.

--- Successfully loaded 4 candidates from Drive ---
  candidate_name                                           raw_text
0       Alice_DS  Skills * Programming Languages: Python (pandas...
1         Bob_DS  Education Details \nMay 2013 to May 2017 B.E  ...
2     Charlie_HR  TECHNICAL SKILLS â¢ Typewriting â¢ TORA â¢ ...
3      Diana_Web  Technical Skills Web Technologies: Angular JS,...


## 1. Text Preprocessing
Standardize raw text by stripping punctuation, converting to lowercase, removing stop words, and lemmatizing.

In [7]:
nltk.download('stopwords', quiet=True)#quite helps to remove output from terminal
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def clean_and_lemmatize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    cleanwords = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(cleanwords)

clean_jd = clean_and_lemmatize(job_desc)
df['clean_text'] = df['raw_text'].apply(clean_and_lemmatize)
print("--- Cleaned Text ---")
print(df[['candidate_name', 'clean_text']])

--- Cleaned Text ---
  candidate_name                                         clean_text
0       Alice_DS  skill programming language python panda numpy ...
1         Bob_DS  education detail may may uitrgpv data scientis...
2     Charlie_HR  technical skill typewriting tora spsseducation...
3      Diana_Web  technical skill web technology angular j html ...


## 2. TF-IDF Vectorization
Convert the cleaned strings into a mathematical matrix.

In [8]:
vectorizer = TfidfVectorizer()
all_text = [clean_jd] + df['clean_text'].tolist()
tfidf_matrix = vectorizer.fit_transform(all_text)

feature_names = vectorizer.get_feature_names_out()
dense_matrix = tfidf_matrix.toarray()

matrix_df = pd.DataFrame(dense_matrix, columns=feature_names)

labels = ['Job Description'] + df['candidate_name'].tolist()
matrix_df.index = labels

print("-----------------TF-IDF Matrix----------------")
print(matrix_df.round(3).T)

-----------------TF-IDF Matrix----------------
                   Job Description  Alice_DS  Bob_DS  Charlie_HR  Diana_Web
accelerating                   0.0     0.030   0.000       0.000      0.000
accounting                     0.0     0.030   0.000       0.000      0.000
achieved                       0.0     0.000   0.000       0.000      0.096
achievementstasks              0.0     0.000   0.078       0.000      0.000
across                         0.0     0.091   0.000       0.000      0.000
...                            ...       ...     ...         ...        ...
wordvec                        0.0     0.030   0.000       0.000      0.000
worked                         0.0     0.020   0.053       0.000      0.021
wwwjalloshbandcom              0.0     0.000   0.000       0.000      0.064
year                           0.0     0.017   0.265       0.083      0.036
young                          0.0     0.061   0.000       0.000      0.000

[517 rows x 5 columns]


## 3. Cosine Similarity, Ranking, & Export
Calculate the geometric angle between the Job Description and Candidate vectors to score alignment, then export to CSV.

In [9]:
from google.colab import drive

# 1. Score
vector = tfidf_matrix[0]
candidate_vectors = tfidf_matrix[1:]
scores = cosine_similarity(vector, candidate_vectors).flatten()

df['similarity_score'] = scores
df_ranked = df.sort_values(by='similarity_score', ascending=False)

print("--- Final Candidate Ranking ---")
print(df_ranked[['candidate_name', 'similarity_score', 'raw_text']])

# 2. Export
drive.mount('/content/drive')
export_df = df_ranked[['candidate_name', 'similarity_score', 'raw_text']]
file_path = "/content/drive/MyDrive/candidate_ranking_results.csv"
export_df.to_csv(file_path, index=False)
print(f"\nResults successfully exported to {file_path}")

--- Final Candidate Ranking ---
  candidate_name  similarity_score  \
0       Alice_DS          0.136778   
1         Bob_DS          0.081915   
3      Diana_Web          0.032673   
2     Charlie_HR          0.022452   

                                            raw_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \nMay 2013 to May 2017 B.E  ...  
3  Technical Skills Web Technologies: Angular JS,...  
2  TECHNICAL SKILLS â¢ Typewriting â¢ TORA â¢ ...  
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Results successfully exported to /content/drive/MyDrive/candidate_ranking_results.csv


## 4. Stage 1 Filter: Supervised Classification
Before ranking, we feed resumes into a Logistic Regression model trained on historical data. This functions as a gatekeeper: it classifies the job category for each candidate and discards bad fits immediately. That saves computing power, so our scoring engine can concentrate solely on qualified candidates.”

In [10]:
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore') # Suppress sklearn convergence warnings for clean output

print("Downloading training dataset...")
# Using the exact HuggingFace dataset URL you provided
dataset_url = "https://huggingface.co/datasets/Unknown92/Resume_dataset/raw/main/UpdatedResumeDataSet.csv"

try:
    # 1. Load the training data
    df_train = pd.read_csv(dataset_url)

    # 2. Clean the training data using our existing function
    print("Cleaning training data (this takes a few seconds)...")
    df_train['cleaned_resume'] = df_train['Resume'].apply(clean_and_lemmatize)

    # 3. Vectorize the training data
    # (We MUST use a separate vectorizer for the classifier, distinct from our ranking vectorizer)
    clf_vectorizer = TfidfVectorizer()
    X_train = clf_vectorizer.fit_transform(df_train['cleaned_resume'])
    y_train = df_train['Category']

    # 4. Train the Logistic Regression Model
    print("Training Logistic Regression classifier...")
    classifier = LogisticRegression(max_iter=1000)
    classifier.fit(X_train, y_train)

    print("✅ Classifier trained successfully on", len(df_train), "resumes.")
    print("Available categories include:", y_train.unique()[:5], "...")

except Exception as e:
    print(f"Failed to load or train on dataset: {e}")

Cleaning training data (this takes a few seconds)...
Training Logistic Regression classifier...
✅ Classifier trained successfully on 962 resumes.
Available categories include: ['Data Science' 'HR' 'Advocate' 'Arts' 'Web Designing'] ...


In [11]:
# Define our target category (must match a category from the HuggingFace dataset)
target_category = "Data Science"

print(f"--- FILTERING PHASE: Target Category = {target_category} ---\n")

# 1. Vectorize our mock candidates using the CLASSIFIER'S vectorizer
X_candidates = clf_vectorizer.transform(df['clean_text'])

# 2. Predict their categories
predicted_categories = classifier.predict(X_candidates)
df['predicted_category'] = predicted_categories

print("Model Predictions:")
for index, row in df.iterrows():
    print(f"{row['candidate_name']}: {row['predicted_category']}")

# 3. The Filter: Keep only candidates matching the target category
df_filtered = df[df['predicted_category'] == target_category].copy()

print(f"\nCandidates surviving the filter: {len(df_filtered)}")

# 4. The Ranker: If anyone survived, rank them using our Stage 2 math
if not df_filtered.empty:
    print("\n--- RANKING PHASE (TF-IDF + Cosine Similarity) ---")

    # We re-vectorize using the RANKING vectorizer just for the survivors
    rank_vectorizer = TfidfVectorizer()
    all_text_to_rank = [clean_jd] + df_filtered['clean_text'].tolist()

    rank_matrix = rank_vectorizer.fit_transform(all_text_to_rank)
    jd_vector = rank_matrix[0]
    survivor_vectors = rank_matrix[1:]

    scores = cosine_similarity(jd_vector, survivor_vectors).flatten()
    df_filtered['similarity_score'] = scores

    final_results = df_filtered.sort_values(by='similarity_score', ascending=False)
    print(final_results[['candidate_name', 'predicted_category', 'similarity_score']])
else:
    print("No candidates matched the required category. Ranking aborted.")

--- FILTERING PHASE: Target Category = Data Science ---

Model Predictions:
Alice_DS: Data Science
Bob_DS: Data Science
Charlie_HR: HR
Diana_Web: Web Designing

Candidates surviving the filter: 2

--- RANKING PHASE (TF-IDF + Cosine Similarity) ---
  candidate_name predicted_category  similarity_score
0       Alice_DS       Data Science          0.125443
1         Bob_DS       Data Science          0.064010
